# Federated Learning Timing and Accuracy Analysis
## Focus: Bottleneck di comunicazione e accuratezza

Notebook strutturato in due parti:
1. Flower Bagging vs Flower Cyclic
2. Flower Bagging vs Flower Cyclic vs NVIDIA FLARE

Obiettivo principale: capire dove si perde tempo (setup, avvio client, training) e come cambia l'accuratezza.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")
%matplotlib inline

ROOT = Path("..").resolve()
BENCHMARKS_DIR = ROOT / "benchmarks"
RESULTS_DIR = ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Se una metrica di accuratezza non e' presente nei file, usa questo fallback.
manual_mae = {
    "flower_bagging": 12.23,
    "flower_cyclic": 11.38,
    "nvidia_flare": np.nan,
}

print(f"Root: {ROOT}")
print(f"Benchmarks: {BENCHMARKS_DIR}")
print(f"Results: {RESULTS_DIR}")

: 

## Parte A - Flower Bagging vs Flower Cyclic

Prima confrontiamo solo i due approcci Flower in termini di timing e accuratezza.

In [ ]:
def load_json_if_exists(path: Path):
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)

bagging_timing = load_json_if_exists(BENCHMARKS_DIR / "flower_bagging" / "results" / "timing_metrics.json")
cyclic_timing = load_json_if_exists(BENCHMARKS_DIR / "flower_cyclic" / "results" / "timing_metrics.json")

records = []
for name, payload in [("flower_bagging", bagging_timing), ("flower_cyclic", cyclic_timing)]:
    if payload is None:
        continue
    total = float(payload.get("total_time", np.nan))
    fl_time = float(payload.get("fl_time", np.nan))
    rec = {
        "approach": name,
        "total_time_s": total,
        "fl_time_s": fl_time,
        "setup_plus_overhead_s": max(total - fl_time, 0.0) if pd.notna(total) and pd.notna(fl_time) else np.nan,
        "avg_round_time_s": float(payload.get("avg_round_time", np.nan)),
        "num_rounds": int(payload.get("num_rounds", np.nan)) if str(payload.get("num_rounds", "")).isdigit() else payload.get("num_rounds"),
    }
    records.append(rec)

df_bc = pd.DataFrame(records)

display(df_bc)
if df_bc.empty:
    print("Nessun timing trovato per Bagging/Cyclic.")

### A1. Confronto tempi globali (Bagging vs Cyclic)

Confronto su tempo totale e tempo medio per round.

In [ ]:
if not df_bc.empty:
    order = df_bc.sort_values("total_time_s")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(data=order, x="total_time_s", y="approach", ax=axes[0])
    axes[0].set_title("Tempo totale")
    axes[0].set_xlabel("Secondi")
    axes[0].set_ylabel("")

    sns.barplot(data=order, x="avg_round_time_s", y="approach", ax=axes[1])
    axes[1].set_title("Tempo medio per round")
    axes[1].set_xlabel("Secondi")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()

    if len(order) == 2:
        fastest = order.iloc[0]
        slowest = order.iloc[1]
        pct = (slowest["total_time_s"] - fastest["total_time_s"]) / slowest["total_time_s"] * 100
        print(f"Piu' veloce: {fastest['approach']} ({fastest['total_time_s']:.2f}s)")
        print(f"Riduzione tempo vs piu' lento: {pct:.1f}%")

### A2. Breakdown fasi e bottleneck (Bagging vs Cyclic)

Con i file timing attuali, la scomposizione diretta disponibile e':
- fl_time_s: durata del ciclo federato
- setup_plus_overhead_s: differenza tra tempo totale e fl_time

In [ ]:
if not df_bc.empty:
    phase_cols = ["fl_time_s", "setup_plus_overhead_s"]
    phase_df = df_bc[["approach"] + phase_cols].set_index("approach")

    phase_df.plot(kind="bar", stacked=True, figsize=(10, 5))
    plt.title("Breakdown temporale Flower")
    plt.ylabel("Secondi")
    plt.xlabel("")
    plt.legend(title="Fase")
    plt.tight_layout()
    plt.show()

    share = phase_df.div(phase_df.sum(axis=1), axis=0) * 100
    print("Quota percentuale per fase (%):")
    display(share.round(2))

## Parte B - Flower vs NVIDIA FLARE

Ora estendiamo a 3 approcci e analizziamo piu' a fondo i tempi NVFLARE (configurazione client, avvio client, eventi all_reduce).

In [ ]:
nvflare_timing = load_json_if_exists(BENCHMARKS_DIR / "nvidia_flare" / "results" / "timing_metrics.json")
log_path = BENCHMARKS_DIR / "nvidia_flare" / "run_nvflare.log"

nvflare_row = None
if nvflare_timing is not None:
    total = float(nvflare_timing.get("total_time", np.nan))
    fl_time = float(nvflare_timing.get("fl_time", np.nan))
    nvflare_row = {
        "approach": "nvidia_flare",
        "total_time_s": total,
        "fl_time_s": fl_time,
        "setup_plus_overhead_s": max(total - fl_time, 0.0) if pd.notna(total) and pd.notna(fl_time) else np.nan,
        "avg_round_time_s": float(nvflare_timing.get("avg_round_time", np.nan)),
        "num_rounds": int(nvflare_timing.get("num_rounds", np.nan)) if str(nvflare_timing.get("num_rounds", "")).isdigit() else nvflare_timing.get("num_rounds"),
    }

phase_metrics = {
    "client_configuration_s": np.nan,
    "client_starting_s": np.nan,
    "allreduce_events": np.nan,
}

if log_path.exists():
    raw_log = log_path.read_text(errors="ignore")
    clean_log = re.sub(r"\x1b\[[0-9;]*m", "", raw_log)

    m_cfg = re.search(r"client configuration took ([0-9]+\.?[0-9]*) seconds", clean_log)
    m_start = re.search(r"client starting took ([0-9]+\.?[0-9]*) seconds", clean_log)
    allreduce_count = len(re.findall(r"Request seq op='allreduce'", clean_log))

    if m_cfg:
        phase_metrics["client_configuration_s"] = float(m_cfg.group(1))
    if m_start:
        phase_metrics["client_starting_s"] = float(m_start.group(1))
    phase_metrics["allreduce_events"] = allreduce_count

if nvflare_row is not None:
    nvflare_row.update(phase_metrics)

df_three = df_bc.copy()
if nvflare_row is not None:
    df_three = pd.concat([df_three, pd.DataFrame([nvflare_row])], ignore_index=True)

display(df_three)

if not df_three.empty and "client_configuration_s" in df_three.columns:
    # stima della parte FL che non e' setup client
    df_three["fl_core_estimated_s"] = np.where(
        df_three["approach"].eq("nvidia_flare"),
        df_three["fl_time_s"] - df_three["client_configuration_s"].fillna(0) - df_three["client_starting_s"].fillna(0),
        np.nan,
    )
    display(df_three[[c for c in ["approach", "client_configuration_s", "client_starting_s", "fl_core_estimated_s", "allreduce_events"] if c in df_three.columns]])

### B1. Tempi globali e colli di bottiglia principali

Confronto a 3 approcci sui tempi globali e zoom sulle fasi NVFLARE.

In [ ]:
if not df_three.empty:
    order = df_three.sort_values("total_time_s")

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.barplot(data=order, x="total_time_s", y="approach", ax=axes[0])
    axes[0].set_title("Tempo totale (3 approcci)")
    axes[0].set_xlabel("Secondi")
    axes[0].set_ylabel("")

    sns.barplot(data=order, x="avg_round_time_s", y="approach", ax=axes[1])
    axes[1].set_title("Tempo medio per round")
    axes[1].set_xlabel("Secondi")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()

if "nvidia_flare" in set(df_three.get("approach", [])):
    row = df_three[df_three["approach"] == "nvidia_flare"].iloc[0]
    phase_parts = {
        "client_configuration_s": row.get("client_configuration_s", np.nan),
        "client_starting_s": row.get("client_starting_s", np.nan),
        "fl_core_estimated_s": row.get("fl_core_estimated_s", np.nan),
    }
    phase_parts = {k: v for k, v in phase_parts.items() if pd.notna(v) and v >= 0}

    if phase_parts:
        plt.figure(figsize=(8, 5))
        plt.bar(phase_parts.keys(), phase_parts.values())
        plt.xticks(rotation=20)
        plt.ylabel("Secondi")
        plt.title("NVFLARE - Breakdown fasi principali")
        plt.tight_layout()
        plt.show()

        total_nv = row["total_time_s"]
        print("NVFLARE bottleneck share (% su tempo totale):")
        for k, v in phase_parts.items():
            print(f"- {k}: {100 * v / total_nv:.1f}%")

        if pd.notna(row.get("allreduce_events", np.nan)):
            print(f"- allreduce events osservati nel log: {int(row['allreduce_events'])}")

### B2. Accuratezza: confronto 2-way e 3-way

Il notebook cerca MAE nei file risultati. Se mancante, usa il fallback manuale definito nella prima cella.

In [ ]:
# Prova a leggere MAE dai csv risultati disponibili
csv_mae = {}
for csv_file in RESULTS_DIR.glob("*.csv"):
    try:
        df_tmp = pd.read_csv(csv_file)
        if "approach" in df_tmp.columns and "final_mae" in df_tmp.columns:
            for _, r in df_tmp.iterrows():
                if pd.notna(r.get("final_mae", np.nan)):
                    csv_mae[str(r["approach"])] = float(r["final_mae"])
    except Exception:
        pass

accuracy_records = []
for approach in ["flower_bagging", "flower_cyclic", "nvidia_flare"]:
    mae_value = csv_mae.get(approach, manual_mae.get(approach, np.nan))
    source = "csv" if approach in csv_mae else "manual_or_missing"
    accuracy_records.append({"approach": approach, "mae": mae_value, "source": source})

df_acc = pd.DataFrame(accuracy_records)
print("MAE disponibili:")
display(df_acc)

plot_acc = df_acc.dropna(subset=["mae"]) 
if not plot_acc.empty:
    plt.figure(figsize=(8, 4))
    order = plot_acc.sort_values("mae")
    sns.barplot(data=order, x="mae", y="approach")
    plt.title("MAE comparison (lower is better)")
    plt.xlabel("MAE")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

    print(f"Migliore MAE attuale: {order.iloc[0]['approach']} ({order.iloc[0]['mae']:.3f})")
else:
    print("Nessun MAE disponibile: aggiorna manual_mae o esporta metrica finale nei CSV.")

## Sintesi Operativa

- Per il confronto tempi: usare principalmente total_time_s, avg_round_time_s e breakdown NVFLARE da log.
- Per i bottleneck nodo-nodo: monitorare client_configuration_s, client_starting_s e numero eventi all_reduce.
- Per l'accuratezza: usare MAE dai CSV quando presente; altrimenti fallback manuale.

Prossimo miglioramento consigliato:
- salvare in modo uniforme final_mae e timing di fase per tutti e 3 gli approcci in JSON/CSV strutturati.

### Checklist prima del report finale

1. Confermare MAE di NVFLARE (se non ancora salvato in results CSV/JSON).
2. Rieseguire il notebook dall'inizio per avere grafici aggiornati.
3. Esportare figure principali:
   - tempi globali 2-way
   - tempi globali 3-way
   - breakdown NVFLARE
   - confronto MAE